# Rocjitsu notebook demo

Run upstream ROCm PyTorch on simulated AMD GPUs from a running notebook. The installation supplies the native runtime, Python worker, and JSON presets.

Install the bundled `rocjitsu` wheel, ROCm-enabled PyTorch, and a compatible Triton compiler into the current kernel's environment, following the [installation instructions](../README.md#python-execution-api), then run all cells. For a CUDA-based kernel, the README explains how to select a separate ROCm worker with `python=`.

In [ ]:
import rocjitsu

rocjitsu.config.names()

## A PyTorch workload

GPU allocations stay in the worker. Return CPU tensors to the notebook.

In [ ]:
def add_one(n):
    import os
    import torch

    x = torch.arange(n, device="cuda", dtype=torch.float32)
    output = (x + 1).cpu()
    torch.testing.assert_close(output, torch.arange(n, dtype=torch.float32) + 1,
                               rtol=0, atol=0)
    return {
        "arch": torch.cuda.get_device_properties(0).gcnArchName,
        "worker_pid": os.getpid(),
        "values": output,
    }

In [ ]:
result = rocjitsu.run(add_one, 257, config=rocjitsu.config.gfx1250_mi455x)
assert result["arch"].split(":")[0] == "gfx1250"
result["arch"], result["values"][:8]

## Reuse a worker

A session keeps imports and runtime state across calls.

In [ ]:
with rocjitsu.Session(rocjitsu.config.gfx1250_mi455x) as sim:
    first = sim.run(add_one, 257)
    second = sim.run(add_one, 129)
    assert first["worker_pid"] == second["worker_pid"] == sim.pid

first["worker_pid"], second["values"][:8]

## Select another installed JSON preset

The names are exactly the existing JSON filename stems.

In [ ]:
result = rocjitsu.run(add_one, 257, config=rocjitsu.config.gfx1201_r9700)
assert result["arch"].split(":")[0] == "gfx1201"
result["arch"], result["values"][:8]